# FAITH-Detect — Full experimental grid (Colab T4)

Runs the same `run_full_experiment` code as the local smoke run, scaled up: 5 seeds,
`roberta-base`, full training data, and **RAID's `reviews` domain** for cross-domain
(same-task) + **cross-generator** evaluation — the strongest generalization evidence.

Steps: GPU check -> deps -> upload code + MAiDE-up CSV -> download RAID `reviews` -> run -> download.

> Runtime -> Change runtime type -> **T4 GPU** before running.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 1) Dependencies.
!pip -q install captum statsmodels umap-learn shap lime datasets spacy huggingface_hub >/dev/null
!python -m spacy download en_core_web_sm -q
import nltk
for p in ['stopwords','punkt','punkt_tab','wordnet','omw-1.4','averaged_perceptron_tagger_eng']:
    nltk.download(p, quiet=True)
print('deps ready')

In [ ]:
# 2) Get the FAITH-Detect code (upload FAITH-Detect.zip from the repo root).
import os, zipfile
from google.colab import files
if not os.path.exists('FAITH-Detect'):
    print('Upload FAITH-Detect.zip:')
    up = files.upload(); name = next(iter(up))
    with zipfile.ZipFile(name) as z: z.extractall('.')
%cd FAITH-Detect
import sys; sys.path.insert(0, 'src')

In [ ]:
# 3) MAiDE-up CSV -> ./data/all_data.csv
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/all_data.csv'):
    from google.colab import files; import shutil
    print('Upload all_data.csv:'); up = files.upload()
    shutil.move(next(iter(up)), 'data/all_data.csv')
print('CSV ready:', os.path.exists('data/all_data.csv'))

In [ ]:
# 4) RAID 'reviews' OOD (cross-domain same-task + cross-generator).
#    RAID's train.csv is domain-sorted, so streaming to 'reviews' is slow; we download the
#    CSV once and chunk-read it (the loader early-stops after passing the reviews domain).
#    NOTE: the CSV is large (tens of GB) and the download + scan take a while on Colab.
#    If you prefer a fast, no-download option, skip this cell and set ood_domains=['abstracts']
#    with ood_csv_path=None in cell 5 (streamed, cheap).
from huggingface_hub import hf_hub_download
from faithdetect.data import load_raid_from_csv
RAID_CSV = hf_hub_download('liamdugan/raid', 'train.csv', repo_type='dataset')
print('RAID CSV at', RAID_CSV)
# Pre-build + cache the reviews OOD frame (humans + all generators, balanced).
ood = load_raid_from_csv(RAID_CSV, domains=('reviews',), cap_per_group=300,
                         cache_path='results/cache/raid_reviews.parquet')
print('reviews OOD:', len(ood), 'rows; generators:', sorted(ood['model'].unique()))

In [ ]:
# 5) Configure and run the full grid (5 seeds, roberta-base, reviews OOD + cross-generator).
from faithdetect.experiment import ExperimentConfig, run_full_experiment
from faithdetect.utils.logging import save_json
from faithdetect.viz import make_all_figures

cfg = ExperimentConfig(
    name='full', data_csv='data/all_data.csv', encoder_name='roberta-base',
    variants=('baseline','hardmask','softreg'), seeds=(0,1,2,3,4),
    epochs=4, batch_size=32, max_length=256, train_subsample=None,
    softreg_lambda=0.5, xai_method='ig', ig_steps=50,
    faithfulness_n_texts=100, n_example_explanations=8,
    # Cross-domain (same task) + cross-generator via the pre-downloaded RAID CSV:
    ood_csv_path=RAID_CSV, ood_domains=('reviews',), ood_cap_per_group=300,
    ood_cache='results/cache/raid_reviews.parquet', ood_per_generator=True,
    device='cuda',
)
results = run_full_experiment(cfg)
save_json('results/full_results.json', results)
figs = make_all_figures(results, 'figures')
print('done:', len(figs), 'figures')

In [ ]:
# 6) Results table + download.
!python scripts/summarize_results.py --results results/full_results.json
!zip -qr faithdetect_outputs.zip results/full_results.json figures
from google.colab import files; files.download('faithdetect_outputs.zip')

### Variations
- **Add a harder cross-domain shift:** `ood_domains=('reviews','news')`.
- **roberta-large ablation:** `encoder_name='roberta-large'` (slower; may need `batch_size=16`).
- **Adversarial robustness:** RAID also ships 11 attacks; load with `attacks=(...)` in
  `load_raid_from_csv` and evaluate as an extra OOD frame.